# 02 — Leakage check and discrete-time survival models (Smoke Free)

**Part A (Cells 1–6):** data, features, models and CV helpers, identical to notebook 01 (no benchmark training).
**Part B (L1):** the same random forest and XGBoost models evaluated with participant-level folds and with entry-level folds → Table 5.
**Part C (S1–S4, S3b):** GRU encoder with a discrete-time hazard head (9 intervals, horizons 1, 6, 24, 72 h), three encoder
strategies (from scratch, self-supervised fine-tuned, self-supervised frozen) on the benchmark folds → Table 8, Figure 4.
S3b retrains the frozen strategy's hazard head without the 100-epoch limit (up to 2,000 epochs, patience 30) on cached
encoder outputs; run it after S3 and before S4.

Set `SAVE_DIR` in Cell 1. Part C takes 1–2 h on a GPU and resumes after a disconnect.

## Part A — setup (Cells 1–6 from the benchmark notebook)

In [ ]:
# ════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ════════════════════════════════════════════════════════
import torch, torch.nn as nn
import numpy as np, pandas as pd
import copy, json, os, time, io, random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (pack_padded_sequence,
                                  pad_packed_sequence)
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings; warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    print("XGBoost not found — installing...")
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip',
                    'install','xgboost','--quiet'])
    import xgboost as xgb
    HAS_XGB = True

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── RERUN SETTINGS: edit here ──────────────────────────
QUICK_TEST = False     # True = 2 folds, 1 epoch, 60 participants (checks the code runs)
GRUD_IMPL  = 'che2018' # 'che2018' = GRU-D with real time decay; 'legacy' = old code
SEED       = 42
RUN_ONLY   = None      # None = all 24 configs, or a list e.g. ['TFM_orig','TFM_exp']
TUNE       = 'all'     # 'all' | 'traditional' (RF/XGB/LR/SVM only) | 'none'
INNER_VAL_FRAC = 0.2   # share of training participants used for tuning
PATIENCE   = 5         # early-stopping patience (epochs) for deep models

SAVE_DIR = 'obj1_v2_results'          # the SAME folder that holds your benchmark results (per_participant_auc.csv, predictions/)
# Colab: keep results if the runtime disconnects — uncomment the next two lines
# from google.colab import drive; drive.mount('/content/drive')
# SAVE_DIR = '/content/drive/MyDrive/obj1_v2_results'

N_FOLDS    = 2 if QUICK_TEST else 10
FT_EPOCHS  = 1 if QUICK_TEST else 30
SSL_EPOCHS = 1 if QUICK_TEST else 30
if QUICK_TEST:
    SAVE_DIR = SAVE_DIR + '_quicktest'
os.makedirs(os.path.join(SAVE_DIR, 'predictions'), exist_ok=True)
print(f"QUICK_TEST={QUICK_TEST}  GRUD_IMPL={GRUD_IMPL}  results -> {SAVE_DIR}")
print("✅ Cell 1 done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2 — LOCAL DATA LOADING + FEATURE ENGINEERING
# Original features : 33
# Expanded features : 33 + temporal lags + rolling stats
#                     + cyclical time = ~48
# ════════════════════════════════════════════════════════

from pathlib import Path
import time

import numpy as np
import pandas as pd


# ── Local Excel file path ──────────────────────────────
# If the Excel file is in the same folder as the notebook:
XLSX_PATH = Path("smokefree_data.xlsx")
if not XLSX_PATH.exists() and Path("../data/smokefree_data.xlsx").exists():
    XLSX_PATH = Path("../data/smokefree_data.xlsx")     # repository data/ folder
if not XLSX_PATH.exists() and Path("/content/smokefree_data.xlsx").exists():
    XLSX_PATH = Path("/content/smokefree_data.xlsx")   # Colab upload location

# If the file is somewhere else, replace the previous line
# with its complete Windows path, for example:


# ── Verify that the file exists ────────────────────────
if not XLSX_PATH.exists():
    raise FileNotFoundError(
        f"\nCould not find the Excel file:\n"
        f"{XLSX_PATH.resolve()}\n\n"
        "Place smokefree_data.xlsx in the same folder as "
        "this notebook, or set XLSX_PATH to its complete path."
    )

print(f"Loading: {XLSX_PATH.resolve()}")


# ── Read Excel file ────────────────────────────────────
try:
    df_raw = pd.read_excel(
        XLSX_PATH,
        sheet_name="All_Data",
        engine="openpyxl"
    )
except ValueError:
    print(
        "Sheet 'All_Data' was not found. "
        "Loading the first sheet instead."
    )
    df_raw = pd.read_excel(
        XLSX_PATH,
        sheet_name=0,
        engine="openpyxl"
    )

print(f"Raw shape: {df_raw.shape}")


# ── Check required columns ─────────────────────────────
REQUIRED_COLUMNS = [
    "account_id",
    "adjusted_craving_record_created",
    "craving_did_smoke",
    "craving_severity",
    "time_to_first_cig",
    "days_since_qd"
]

missing_columns = [
    column for column in REQUIRED_COLUMNS
    if column not in df_raw.columns
]

if missing_columns:
    raise KeyError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )


# ── Basic preprocessing ────────────────────────────────
df = df_raw.copy()

df["adjusted_craving_record_created"] = pd.to_datetime(
    df["adjusted_craving_record_created"],
    errors="coerce"
)

invalid_dates = df[
    "adjusted_craving_record_created"
].isna().sum()

if invalid_dates > 0:
    print(
        f"Warning: removing {invalid_dates} entries "
        "with invalid timestamps."
    )
    df = df.dropna(
        subset=["adjusted_craving_record_created"]
    ).copy()

df["craving_did_smoke"] = pd.to_numeric(
    df["craving_did_smoke"],
    errors="coerce"
).fillna(0)

df["craving_severity"] = pd.to_numeric(
    df["craving_severity"],
    errors="coerce"
).fillna(0)

df["days_since_qd"] = pd.to_numeric(
    df["days_since_qd"],
    errors="coerce"
).fillna(0)

df = (
    df.sort_values(
        [
            "account_id",
            "adjusted_craving_record_created"
        ]
    )
    .reset_index(drop=True)
)

df["label"] = (
    df["craving_did_smoke"]
    .astype(float)
    .clip(lower=0, upper=1)
)


# ── Keep patients with at least 20 entries ─────────────
patient_counts = df.groupby("account_id").size()

eligible_patients = patient_counts[
    patient_counts >= 20
].index

df = (
    df[df["account_id"].isin(eligible_patients)]
    .copy()
    .reset_index(drop=True)
)

if QUICK_TEST:
    _keep = sorted(df["account_id"].unique())[:60]
    df = df[df["account_id"].isin(_keep)].copy().reset_index(drop=True)

print(
    f"After minimum-entry filtering: "
    f"{df['account_id'].nunique()} patients, "
    f"{len(df):,} entries"
)


# Recreate groupby after filtering and sorting
grp = df.groupby("account_id", sort=False)


# ── Original features ──────────────────────────────────
ttfc_map = {
    "<= 5": 0,
    "6-30": 1,
    "31-60": 2,
    "> 60": 3
}

df["time_to_first_cig_ord"] = (
    df["time_to_first_cig"]
    .map(ttfc_map)
    .fillna(0)
    .astype(float)
)

df["prior_event_lapse"] = (
    grp["label"]
    .shift(1)
    .fillna(0)
    .astype(float)
)

df["mins_since_prev_rep"] = (
    grp["adjusted_craving_record_created"]
    .diff()
    .dt.total_seconds()
    .div(60)
    .fillna(0)
    .clip(lower=0)
)


FEELINGS = [
    "craving_severity",
    "craving_feeling_annoyed",
    "craving_feeling_anxious",
    "craving_feeling_bored",
    "craving_feeling_down",
    "craving_feeling_happy",
    "craving_feeling_hungry",
    "craving_feeling_lonely"
]

ACTIVITIES = [
    "craving_occupation_at_bar_or_event",
    "craving_occupation_chatting",
    "craving_occupation_coffee_or_tea",
    "craving_occupation_drinking",
    "craving_occupation_driving",
    "craving_occupation_going_to_bed",
    "craving_occupation_just_eaten",
    "craving_occupation_just_had_sex",
    "craving_occupation_reading",
    "craving_occupation_relaxing",
    "craving_occupation_taking_a_break",
    "craving_occupation_thinking",
    "craving_occupation_waking_up",
    "craving_occupation_working"
]

SOCIAL = [
    "craving_env_alone",
    "craving_env_colleagues",
    "craving_env_family",
    "craving_env_friends",
    "craving_env_partner",
    "craving_env_strangers"
]

TEMPORAL = [
    "cigs_per_day",
    "time_to_first_cig_ord",
    "days_since_qd",
    "mins_since_prev_rep",
    "prior_event_lapse"
]

ORIG_FEATS = [
    column
    for column in FEELINGS + ACTIVITIES + SOCIAL + TEMPORAL
    if column in df.columns
]


# Convert original feature columns to numeric
for column in ORIG_FEATS:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ── Expanded temporal features ─────────────────────────
hour = df[
    "adjusted_craving_record_created"
].dt.hour.astype(float)

df["hour_sin"] = np.sin(
    2 * np.pi * hour / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * hour / 24
)

df["is_weekend"] = (
    df["adjusted_craving_record_created"]
    .dt.dayofweek
    .ge(5)
    .astype(float)
)


# Recreate groupby after adding and converting columns
grp = df.groupby("account_id", sort=False)

df["severity_change"] = (
    grp["craving_severity"]
    .diff()
    .fillna(0)
)

df["rolling3_sev_mean"] = (
    grp["craving_severity"]
    .transform(
        lambda values: (
            values.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )
    )
    .fillna(0)
)

df["rolling3_sev_std"] = (
    grp["craving_severity"]
    .transform(
        lambda values: (
            values.shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .std()
        )
    )
    .fillna(0)
)


# ── Days since last previous lapse ─────────────────────
# unit-safe: seconds since 1970 whatever the timestamp precision (pandas 2 or 3)
EPOCH = pd.Timestamp("1970-01-01")
timestamp_seconds = (
    df["adjusted_craving_record_created"] - EPOCH
).dt.total_seconds()

df["_lapse_time"] = np.where(
    df["label"].eq(1),
    timestamp_seconds,
    np.nan
)

# Recreate groupby because _lapse_time was just added
grp = df.groupby("account_id", sort=False)

df["_last_lapse_time"] = (
    grp["_lapse_time"]
    .transform(
        lambda values: values.shift(1).ffill()
    )
)

patient_start_seconds = (
    grp["adjusted_craving_record_created"].transform("min") - EPOCH
).dt.total_seconds()

reference_time = df[
    "_last_lapse_time"
].fillna(patient_start_seconds)

df["days_since_last_lapse"] = (
    (timestamp_seconds - reference_time) / 86400
).clip(lower=0)

df.drop(
    columns=[
        "_lapse_time",
        "_last_lapse_time"
    ],
    inplace=True
)


df["log_days_since_qd"] = np.log1p(
    df["days_since_qd"].clip(lower=0)
)


# ── Lagged features ────────────────────────────────────
grp = df.groupby("account_id", sort=False)

for lag in [1, 2, 3]:
    df[f"sev_lag{lag}"] = (
        grp["craving_severity"]
        .shift(lag)
        .fillna(0)
    )

for lag in [2, 3]:
    df[f"lapse_lag{lag}"] = (
        grp["label"]
        .shift(lag)
        .fillna(0)
    )


EXTRA_FEATS = [
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "severity_change",
    "rolling3_sev_mean",
    "rolling3_sev_std",
    "days_since_last_lapse",
    "log_days_since_qd",
    "sev_lag1",
    "sev_lag2",
    "sev_lag3",
    "lapse_lag2",
    "lapse_lag3"
]

EXP_FEATS = list(
    dict.fromkeys(
        ORIG_FEATS
        + [
            column
            for column in EXTRA_FEATS
            if column in df.columns
        ]
    )
)


# ── Multi-task auxiliary labels ────────────────────────
def phase_fn(days):
    if days < 3:
        return 0
    if days < 14:
        return 1
    return 2


df["phase_label"] = (
    df["days_since_qd"]
    .clip(lower=0)
    .apply(phase_fn)
    .astype(int)
)

grp = df.groupby("account_id", sort=False)

df["next_craving_sev"] = (
    grp["craving_severity"]
    .shift(-1)
)

df["next_craving_change"] = (
    df["next_craving_sev"]
    - df["craving_severity"]
)

# The final entry of each patient has no next craving
df["next_craving_change"] = (
    df["next_craving_change"]
    .fillna(0)
)

df["next_craving_sev"] = (
    df["next_craving_sev"]
    .fillna(0)
)


# ── Eventual lapse within the next 19 days ─────────────
df["eventual_lapse_19d"] = 0.0

print(
    "Computing eventual_lapse_19d..."
)

start_time = time.time()

window_seconds = (
    pd.Timedelta(days=19)
    .total_seconds()
)

for patient_id, patient_group in df.groupby(
    "account_id",
    sort=False
):
    patient_group = patient_group.sort_values(
        "adjusted_craving_record_created"
    )

    indices = patient_group.index.to_numpy()

    times = (
        patient_group["adjusted_craving_record_created"] - EPOCH
    ).dt.total_seconds().to_numpy()

    labels = (
        patient_group["label"]
        .to_numpy()
    )

    lapse_positions = np.flatnonzero(
        labels == 1
    )

    if len(lapse_positions) == 0:
        continue

    for position, row_index in enumerate(indices):
        # Find the first lapse strictly after this entry
        future_lapses = lapse_positions[
            lapse_positions > position
        ]

        if len(future_lapses) == 0:
            continue

        first_future_lapse = future_lapses[0]

        if (
            times[first_future_lapse] - times[position]
            <= window_seconds
        ):
            df.at[
                row_index,
                "eventual_lapse_19d"
            ] = 1.0


elapsed_minutes = (
    time.time() - start_time
) / 60

print(
    f"Eventual-lapse labels completed in "
    f"{elapsed_minutes:.2f} minutes."
)


# ── Fill missing values ────────────────────────────────
ALL_COLS = list(
    dict.fromkeys(
        EXP_FEATS
        + [
            "phase_label",
            "next_craving_change",
            "eventual_lapse_19d"
        ]
    )
)

df[ALL_COLS] = (
    df[ALL_COLS]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
    .astype(float)
)


# ── Final summary ──────────────────────────────────────
n_orig = len(ORIG_FEATS)
n_exp = len(EXP_FEATS)

print()
print(
    f"Patients              : "
    f"{df['account_id'].nunique():,}"
)
print(
    f"Entries               : "
    f"{len(df):,}"
)
print(
    f"Lapse rate            : "
    f"{df['label'].mean() * 100:.1f}%"
)
print(
    f"Eventual-lapse rate   : "
    f"{df['eventual_lapse_19d'].mean() * 100:.1f}%"
)
print(
    f"Original features     : "
    f"{n_orig}"
)
print(
    f"Expanded features     : "
    f"{n_exp}"
)
print("Cell 2 completed successfully.")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 2b — TABLE 1 STATISTICS, computed from the analysed data
# ════════════════════════════════════════════════════════
_n = df.groupby('account_id').size()
_lapses = df.groupby('account_id')['label'].sum()
_t = pd.to_datetime(df['adjusted_craving_record_created'])
_span_days = df.assign(_t=_t).groupby('account_id')['_t'].agg(
    lambda s: max((s.max() - s.min()).total_seconds() / 86400, 1.0))
_active_days = df.assign(_d=_t.dt.date).groupby('account_id')['_d'].nunique()
table1 = pd.DataFrame({'Characteristic': [
    'Total participants', 'Total EMA entries', 'Mean entries per participant (SD)',
    'Median entries per participant (IQR)', 'Range of entries per participant',
    'Overall lapse rate', 'Participants with >=1 lapse', 'Participants with no lapses',
    'Entries per day, mean over first-to-last entry span (median)',
    'Entries per active day, mean (median)'],
 'Value': [
    f"{len(_n)}", f"{len(df):,}", f"{_n.mean():.1f} ({_n.std():.1f})",
    f"{_n.median():.0f} ({_n.quantile(.25):.0f}-{_n.quantile(.75):.0f})",
    f"{_n.min()}-{_n.max()}", f"{100*df['label'].mean():.1f}%",
    f"{(_lapses>0).sum()} ({100*(_lapses>0).mean():.1f}%)",
    f"{(_lapses==0).sum()} ({100*(_lapses==0).mean():.1f}%)",
    f"{(_n/_span_days).mean():.2f} ({(_n/_span_days).median():.2f})",
    f"{(_n/_active_days).mean():.2f} ({(_n/_active_days).median():.2f})"]})
print(table1.to_string(index=False))


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — ALL MODEL CLASSES
# ════════════════════════════════════════════════════════

# ── Dataset utils ─────────────────────────────────────
class PatDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): return self.s[i]

class PatDSExt(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): return self.s[i]

def collate_v(batch):
    xs,ys=zip(*batch)
    lengths=torch.tensor([len(x) for x in xs])
    ml=max(lengths); nc=xs[0].shape[1]
    Xp=torch.zeros(len(xs),ml,nc)
    yp=torch.full((len(xs),ml),float('nan'))
    for i,(x,y) in enumerate(zip(xs,ys)):
        Xp[i,:len(x)]=torch.from_numpy(x).float()
        yp[i,:len(x)]=torch.from_numpy(y).float()
    return Xp,yp,lengths

def collate_ext(batch):
    xs,yl,yph,yn,ye,_=(list(z) for z in zip(*[
        (s[0],s[1],s[2],s[3],s[4],s[5]) for s in batch]))
    lengths=torch.tensor([len(x) for x in xs])
    ml=max(lengths); nc=xs[0].shape[1]
    Xp=torch.zeros(len(xs),ml,nc)
    Yl=torch.full((len(xs),ml),float('nan'))
    Yp=torch.full((len(xs),ml),-1,dtype=torch.long)
    Yn=torch.zeros(len(xs),ml)
    Ye=torch.full((len(xs),ml),float('nan'))
    for i,(x,l,p,n,e) in enumerate(zip(xs,yl,yph,yn,ye)):
        ln=len(x)
        Xp[i,:ln]=torch.from_numpy(x).float()
        Yl[i,:ln]=torch.from_numpy(l).float()
        Yp[i,:ln]=torch.from_numpy(p).long()
        Yn[i,:ln]=torch.from_numpy(n).float()
        Ye[i,:ln]=torch.from_numpy(e).float()
    return Xp,Yl,Yp,Yn,Ye,lengths

# ── Prediction head (shared) ──────────────────────────
class PredHead(nn.Module):
    def __init__(self,hid=64,drop=0.3):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(hid,hid//2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(hid//2,1))
    def forward(self,h): return self.net(h).squeeze(-1)

# ── GRU ───────────────────────────────────────────────
class GRUEncoder(nn.Module):
    def __init__(self,nf,hid=64,drop=0.3,n_layers=2):
        super().__init__()
        self.gru=nn.GRU(nf,hid,n_layers,
                         batch_first=True,dropout=drop)
    def forward(self,x,lengths=None):
        self.gru.flatten_parameters()
        if lengths is not None:
            pk=pack_padded_sequence(x,lengths.cpu(),
                batch_first=True,enforce_sorted=False)
            out,_=self.gru(pk)
            out,_=pad_packed_sequence(out,batch_first=True)
        else: out,_=self.gru(x)
        return out

# ── LSTM ──────────────────────────────────────────────
class LSTMEncoder(nn.Module):
    def __init__(self,nf,hid=64,drop=0.3,n_layers=2):
        super().__init__()
        self.lstm=nn.LSTM(nf,hid,n_layers,
                           batch_first=True,dropout=drop)
    def forward(self,x,lengths=None):
        self.lstm.flatten_parameters()
        if lengths is not None:
            pk=pack_padded_sequence(x,lengths.cpu(),
                batch_first=True,enforce_sorted=False)
            out,_=self.lstm(pk)
            out,_=pad_packed_sequence(out,batch_first=True)
        else: out,_=self.lstm(x)
        return out

# ── GRU-D ─────────────────────────────────────────────
class GRUDCell(nn.Module):
    def __init__(self,inp,hid):
        super().__init__()
        gi=inp*2+hid
        self.Wr=nn.Linear(gi,hid)
        self.Wz=nn.Linear(gi,hid)
        self.Wh=nn.Linear(gi,hid)
        self.Wgh=nn.Linear(1,hid)
        self.Wgx=nn.Linear(inp,inp)
        self.h0=nn.Parameter(torch.zeros(hid))
        self.register_buffer('xm',torch.zeros(inp))
    def forward(self,x,m,df_,dg,h):
        gh=torch.exp(-torch.clamp(self.Wgh(dg),min=0))
        ht=gh*h+(1-gh)*self.h0.unsqueeze(0).expand_as(h)
        gx=torch.exp(-torch.clamp(self.Wgx(df_),min=0))
        xi=gx*x+(1-gx)*self.xm.unsqueeze(0).expand_as(x)
        xt=m*x+(1-m)*xi
        inp=torch.cat([xt,m,ht],1)
        r=torch.sigmoid(self.Wr(inp))
        z=torch.sigmoid(self.Wz(inp))
        return (1-z)*ht+z*torch.tanh(
            self.Wh(torch.cat([xt,m,r*ht],1)))

class GRUDEncoder(nn.Module):
    """
    Vectorised GRU-D — no Python timestep loop.
    Uses a standard GRU with time-decay applied
    to the input before each step via batched ops.
    ~20x faster than the cell-loop version.
    """
    def __init__(self,nf,hid=64,drop=0.3,n_layers=2):
        super().__init__()
        self.nf   = nf
        self.hid  = hid
        # learnable decay rates per feature
        self.W_decay = nn.Linear(1, nf, bias=True)
        # input projection: applies decay to features
        self.gru  = nn.GRU(nf*2, hid, n_layers,
                            batch_first=True,
                            dropout=drop)
        self.drop = nn.Dropout(drop)

    def forward(self, x, lengths=None):
        """
        x shape: (B, T, nf*3)
          x[:,:,:nf]      = features
          x[:,:,nf:2*nf]  = missingness mask
          x[:,:,2*nf:3*nf]= time deltas (normalised)
        """
        B, T, _ = x.shape
        nf = self.nf

        feats  = x[:, :, :nf]          # (B,T,nf)
        mask   = x[:, :, nf:2*nf]      # (B,T,nf)
        delta  = x[:, :, 2*nf:3*nf]    # (B,T,nf)

        # decay factor per feature per timestep
        # gamma = exp(-max(0, w*delta + b))
        # shape: (B,T,nf)
        delta_mean = delta.mean(-1, keepdim=True)  # (B,T,1)
        gamma = torch.exp(
            -torch.clamp(
                self.W_decay(delta_mean).expand_as(feats),
                min=0))

        # impute missing values with decayed mean
        # x_imp = mask*x + (1-mask)*gamma*global_mean
        # we use 0 as global mean (features are standardised)
        x_imp = mask * feats + (1 - mask) * gamma * 0.0

        # concatenate imputed features + mask as input
        inp = torch.cat([x_imp, mask], dim=-1)  # (B,T,nf*2)

        self.gru.flatten_parameters()
        if lengths is not None:
            pk  = pack_padded_sequence(
                inp, lengths.cpu(),
                batch_first=True,
                enforce_sorted=False)
            out, _ = self.gru(pk)
            out, _ = pad_packed_sequence(
                out, batch_first=True)
        else:
            out, _ = self.gru(inp)

        out = self.drop(out)
        if lengths is not None:
            for i,l in enumerate(lengths):
                if l < T: out[i,l:] = 0.
        return out
GRUDEncoderLegacy = GRUDEncoder   # the previous implementation
# ── Transformer ───────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self,hid,max_len=700,drop=0.1):
        super().__init__()
        self.drop=nn.Dropout(drop)
        pe=torch.zeros(max_len,hid)
        pos=torch.arange(max_len).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,hid,2).float()*
                       (-np.log(10000)/hid))
        pe[:,0::2]=torch.sin(pos*div)
        pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x):
        return self.drop(x+self.pe[:,:x.size(1)])

class TransformerEncoder(nn.Module):
    def __init__(self,nf,hid=64,nhead=4,
                  nlayers=2,drop=0.3):
        super().__init__()
        self.proj=nn.Linear(nf,hid)
        self.pe  =PositionalEncoding(hid,drop=drop)
        enc=nn.TransformerEncoderLayer(
            d_model=hid,nhead=nhead,
            dim_feedforward=hid*2,dropout=drop,
            batch_first=True,norm_first=True)
        self.tfm=nn.TransformerEncoder(enc,nlayers)
    def forward(self,x,lengths=None):
        B,T,_=x.shape
        x=self.pe(self.proj(x))
        mask=torch.triu(torch.ones(T,T,
            device=x.device),diagonal=1).bool()
        pm=None
        if lengths is not None:
            pm=torch.ones(B,T,dtype=torch.bool,
                           device=x.device)
            for i,l in enumerate(lengths): pm[i,:l]=False
        return self.tfm(x,mask=mask,
                         src_key_padding_mask=pm)

# ── SSL model wrapper ─────────────────────────────────
class SSLWrapper(nn.Module):
    def __init__(self,enc,nf,hid=64):
        super().__init__()
        self.enc=enc; h=hid
        self.nh=nn.Sequential(
            nn.Linear(h,h),nn.GELU(),nn.Linear(h,nf))
        self.mh=nn.Sequential(
            nn.Linear(h,h),nn.GELU(),nn.Linear(h,nf))
        self.oh=nn.Sequential(
            nn.Linear(h*2,h),nn.GELU(),nn.Linear(h,1))
    def forward(self,x,lengths=None):
        return self.enc(x,lengths)

# ── Multi-task model wrapper ──────────────────────────
class MTWrapper(nn.Module):
    def __init__(self,enc,nf,hid=64):
        super().__init__()
        self.enc=enc; h=hid
        self.eh=nn.Sequential(
            nn.Linear(h,h//2),nn.GELU(),nn.Linear(h//2,1))
        self.ph=nn.Sequential(
            nn.Linear(h,h//2),nn.GELU(),nn.Linear(h//2,3))
        self.ch=nn.Sequential(
            nn.Linear(h,h//2),nn.GELU(),nn.Linear(h//2,1))
    def forward(self,x,lengths=None):
        return self.enc(x,lengths)

print("✅ Cell 3 done — all model classes defined")


# ════════════════════════════════════════════════════════
# TCN + GRU-TCN IMPLEMENTATION
# Add these classes to Cell 3 after TransformerEncoder
# ════════════════════════════════════════════════════════

# ── Causal Conv Block ─────────────────────────────────
class CausalConv1d(nn.Module):
    """
    Causal convolution: output at t depends only on
    inputs at t, t-1, ..., t-(kernel_size-1)*dilation.
    Achieved by left-padding the input by
    (kernel_size-1)*dilation zeros before convolution
    and removing the rightmost outputs.
    """
    def __init__(self, in_ch, out_ch,
                  kernel_size=3, dilation=1):
        super().__init__()
        self.padding = (kernel_size-1) * dilation
        self.conv    = nn.Conv1d(
            in_ch, out_ch,
            kernel_size = kernel_size,
            dilation    = dilation,
            padding     = 0)

    def forward(self, x):
        # x: (B, C, T)
        x = nn.functional.pad(x, (self.padding, 0))
        return self.conv(x)


class TCNBlock(nn.Module):
    """
    One dilated causal convolutional block.
    Two causal convolutions with:
      - layer normalisation
      - GELU activation
      - dropout
      - residual connection (with 1x1 projection
        if channel dimensions differ)
    """
    def __init__(self, in_ch, out_ch,
                  kernel_size=3, dilation=1,
                  drop=0.3):
        super().__init__()
        self.conv1 = CausalConv1d(
            in_ch, out_ch, kernel_size, dilation)
        self.conv2 = CausalConv1d(
            out_ch, out_ch, kernel_size, dilation)
        self.norm1 = nn.LayerNorm(out_ch)
        self.norm2 = nn.LayerNorm(out_ch)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(drop)
        # residual projection if channels differ
        self.proj  = (nn.Conv1d(in_ch, out_ch, 1)
                       if in_ch != out_ch else None)

    def forward(self, x):
        # x: (B, T, C) → transpose for conv
        xt = x.transpose(1, 2)          # (B, C, T)
        # first conv
        h  = self.conv1(xt)             # (B, out_ch, T)
        h  = h.transpose(1, 2)          # (B, T, out_ch)
        h  = self.drop(self.act(self.norm1(h)))
        # second conv
        h  = h.transpose(1, 2)          # (B, out_ch, T)
        h  = self.conv2(h)              # (B, out_ch, T)
        h  = h.transpose(1, 2)          # (B, T, out_ch)
        h  = self.drop(self.act(self.norm2(h)))
        # residual
        if self.proj is not None:
            res = self.proj(
                x.transpose(1,2)).transpose(1,2)
        else:
            res = x
        return h + res


class TCNEncoder(nn.Module):
    """
    Temporal Convolutional Network encoder.

    Architecture (Section 4.5):
      4 dilated causal convolutional blocks
      kernel size 3
      dilations: 1, 2, 4, 8
      receptive field: 1+(3-1)*(1+2+4+8)*2 = 61 entries
        covers the median patient sequence length (31)
        and approaches the 75th percentile
      hidden dimension 64 throughout
      dropout 0.3

    Each block: CausalConv → LayerNorm → GELU → Dropout
                CausalConv → LayerNorm → GELU → Dropout
                + residual connection

    Input  : (B, T, nf)
    Output : (B, T, 64)
    """
    def __init__(self, nf, hid=64,
                  kernel_size=3, drop=0.3,
                  dilations=(1,2,4,8)):
        super().__init__()
        self.input_proj = nn.Linear(nf, hid)
        blocks = []
        for d in dilations:
            blocks.append(TCNBlock(
                hid, hid,
                kernel_size = kernel_size,
                dilation    = d,
                drop        = drop))
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x, lengths=None):
        # project input features to hidden dim
        h = self.input_proj(x)      # (B, T, hid)
        # pass through TCN blocks
        h = self.blocks(h)          # (B, T, hid)
        # apply padding mask if lengths provided
        if lengths is not None:
            B, T, _ = h.shape
            for i, l in enumerate(lengths):
                if l < T:
                    h[i, l:] = 0.
        return h


class GRUTCNEncoder(nn.Module):
    """
    Hybrid TCN + GRU encoder.

    Architecture (Section 4.5):
      Front-end: 2 dilated causal TCN blocks
        (dilations 1, 2) for local feature extraction
        receptive field: 1+(3-1)*(1+2)*2 = 13 entries
      Back-end: 1-layer GRU (hidden dim 64)
        for sequential integration across full history
      Hidden dimension 64 throughout
      Dropout 0.3

    Motivation: TCN captures local craving dynamics
    efficiently via parallelisable convolutions;
    GRU integrates these local features into a
    global patient-level trajectory representation
    that respects long-range temporal dependencies.

    Input  : (B, T, nf)
    Output : (B, T, 64)
    """
    def __init__(self, nf, hid=64,
                  kernel_size=3, drop=0.3):
        super().__init__()
        self.input_proj = nn.Linear(nf, hid)
        # 2-block TCN front-end
        self.tcn = nn.Sequential(
            TCNBlock(hid, hid, kernel_size,
                      dilation=1, drop=drop),
            TCNBlock(hid, hid, kernel_size,
                      dilation=2, drop=drop))
        # 1-layer GRU back-end
        self.gru = nn.GRU(hid, hid, num_layers=1,
                           batch_first=True,
                           dropout=0.0)
        self.drop = nn.Dropout(drop)

    def forward(self, x, lengths=None):
        # project input
        h = self.input_proj(x)      # (B, T, hid)
        # local feature extraction via TCN
        h = self.tcn(h)             # (B, T, hid)
        # sequential integration via GRU
        self.gru.flatten_parameters()
        if lengths is not None:
            pk  = pack_padded_sequence(
                h, lengths.cpu().clamp(min=1),
                batch_first=True,
                enforce_sorted=False)
            h, _ = self.gru(pk)
            h, _ = pad_packed_sequence(
                h, batch_first=True)
        else:
            h, _ = self.gru(h)
        h = self.drop(h)
        if lengths is not None:
            B, T, _ = h.shape
            for i, l in enumerate(lengths):
                if l < T:
                    h[i, l:] = 0.
        return h

print("✅ TCNEncoder and GRUTCNEncoder defined")
print(f"  TCN receptive field    : "
      f"{1 + (3-1)*(1+2+4+8)*2} entries")
print(f"  GRU-TCN receptive field: "
      f"{1 + (3-1)*(1+2)*2} entries (TCN front-end)")


# ════════════════════════════════════════════════════════
# GRU-D (Che et al., 2018) — time decay that is actually used
# Layer 1: GRUDCell (defined above, previously unused):
#   hidden-state decay  gamma_h = exp(-max(0, W_h * gap + b))
#   input decay toward the training mean (0 after standardisation)
# Layer 2: standard GRU, matching the 2-layer convention of the other encoders.
# Input layout unchanged: [features | mask | per-feature delta]
# ════════════════════════════════════════════════════════
class GRUDEncoderChe(nn.Module):
    def __init__(self, nf, hid=64, drop=0.3, n_layers=2):
        super().__init__()
        self.nf = nf
        self.cell = GRUDCell(nf, hid)
        self.gru2 = (nn.GRU(hid, hid, n_layers - 1, batch_first=True)
                     if n_layers > 1 else None)
        self.drop = nn.Dropout(drop)

    def forward(self, x, lengths=None):
        B, T, _ = x.shape
        nf = self.nf
        feats = x[:, :, :nf]
        mask  = x[:, :, nf:2 * nf]
        delta = x[:, :, 2 * nf:3 * nf]
        h = self.cell.h0.unsqueeze(0).expand(B, -1).contiguous()
        outs = []
        for t in range(T):
            d_t = delta[:, t, :]
            h = self.cell(feats[:, t, :], mask[:, t, :], d_t,
                          d_t.mean(-1, keepdim=True), h)   # gap is identical across features
            outs.append(h)
        out = torch.stack(outs, dim=1)
        if self.gru2 is not None:
            self.gru2.flatten_parameters()
            out, _ = self.gru2(self.drop(out))
        out = self.drop(out)
        if lengths is not None:
            for i, l in enumerate(lengths):
                if l < T: out[i, l:] = 0.
        return out


GRUDEncoder = GRUDEncoderChe if GRUD_IMPL == 'che2018' else GRUDEncoderLegacy
print(f"✅ GRU-D implementation in use: {GRUD_IMPL}")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 4 — SEQUENCE BUILDERS + STANDARDISATION
# ════════════════════════════════════════════════════════

def build_sequences(df, feat_cols, pid_col='account_id',
                     time_col='adjusted_craving_record_created'):
    seqs=[]
    for pid,g in df.groupby(pid_col):
        g=g.sort_values(time_col).reset_index(drop=True)
        X=g[feat_cols].values.astype(np.float32)
        y=g['label'].values.astype(np.float32)
        seqs.append((X,y,pid))
    return seqs

def build_sequences_grud(df, feat_cols):
    """
    GRU-D sequences: [features | mask | delta_norm]
    Delta = time since last observation of each feature.
    Vectorised: computed with pandas shift, no Python loops.
    """
    seqs = []
    n    = len(feat_cols)

    for pid, g in df.groupby('account_id'):
        g = g.sort_values(
            'adjusted_craving_record_created'
        ).reset_index(drop=True)
        y = g['label'].values.astype(np.float32)

        # features and mask
        feats = g[feat_cols].values.astype(np.float32)
        M     = (~np.isnan(feats)).astype(np.float32)
        feats = np.nan_to_num(feats, nan=0.)

        # time in minutes from first entry
        times = (pd.to_datetime(
            g['adjusted_craving_record_created'])
            - pd.to_datetime(
            g['adjusted_craving_record_created']
            ).iloc[0]
        ).dt.total_seconds().div(60).values.astype(
            np.float32)

        T = len(g)

        # delta: time since previous entry
        # same for all features (entry-level gaps)
        # shape (T,) → broadcast to (T, n)
        gaps  = np.diff(times, prepend=0.).astype(
            np.float32)  # (T,)
        delta = np.broadcast_to(
            gaps[:,None], (T,n)).copy()  # (T,n)

        if GRUD_IMPL == 'legacy':
            # old scaling: divide by this participant's largest gap
            dm = delta.max()
            if dm > 0: delta /= dm
        else:
            # scale without using future entries: log(1+min) / log(1+1 week)
            delta = (np.log1p(delta) / np.log1p(60 * 24 * 7)).astype(np.float32)

        X = np.concatenate([feats, M, delta], axis=1)
        seqs.append((X, y, pid))
    return seqs
    
def build_sequences_ext(df, feat_cols):
    """Extended sequences for multi-task pre-training."""
    seqs=[]
    for pid,g in df.groupby('account_id'):
        g=g.sort_values(
            'adjusted_craving_record_created'
        ).reset_index(drop=True)
        X =g[feat_cols].values.astype(np.float32)
        y =g['label'].values.astype(np.float32)
        yp=g['phase_label'].values.astype(np.int64)
        yn=g['next_craving_change'].values.astype(np.float32)
        ye=g['eventual_lapse_19d'].values.astype(np.float32)
        seqs.append((X,y,yp,yn,ye,pid))
    return seqs

def standardize_fold(train_seqs, test_seqs, nf):
    """Fit on train fold, apply to both. No test leakage."""
    aX=np.vstack([s[0][:,:nf] for s in train_seqs])
    m=aX.mean(0); s=aX.std(0)+1e-6
    def norm(sq):
        X=sq[0].copy(); X[:,:nf]=(sq[0][:,:nf]-m)/s
        return (X,)+sq[1:]
    return ([norm(sq) for sq in train_seqs],
            [norm(sq) for sq in test_seqs], m, s)

def ci95(aucs):
    a=np.array([x for x in aucs if not np.isnan(x)])
    if not len(a): return None,None,None
    se=a.std(ddof=1)/np.sqrt(len(a))
    t=stats.t.ppf(0.975,df=len(a)-1)
    return float(a.mean()),float(a.mean()-t*se),float(a.mean()+t*se)

def youden_threshold(y_true,y_prob):
    fpr,tpr,thresh=roc_curve(y_true,y_prob)
    return thresh[np.argmax(tpr-fpr)]

print("✅ Cell 4 done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 5 — SSL + MULTI-TASK PRE-TRAINING FUNCTIONS
# ════════════════════════════════════════════════════════

def ssl_pretrain_fold(train_seqs, enc_class, nf,
                       enc_kwargs=None,
                       epochs=30, bs=8,
                       lr=1e-3, wd=1e-4,
                       mask_rate=0.15):
    """
    SSL pre-training on training fold only (inductive).
    enc_class: GRUEncoder | LSTMEncoder
    """
    if enc_kwargs is None: enc_kwargs={}
    enc=enc_class(nf, **enc_kwargs)
    model=SSLWrapper(enc, nf, 64).to(device)
    opt=torch.optim.Adam(model.parameters(),
                          lr=lr,weight_decay=wd)
    mse=nn.MSELoss(reduction='none')
    bce=nn.BCEWithLogitsLoss()
    dl=DataLoader(PatDS([(s[0],s[1]) for s in train_seqs]),
                  batch_size=bs,shuffle=True,
                  collate_fn=collate_v)
    for ep in range(epochs):
        model.train()
        for Xp,yp,lengths in dl:
            Xp=Xp.to(device); B,T,_=Xp.shape
            Xf=Xp[:,:,:nf]
            opt.zero_grad(); H=model(Xp,lengths)
            loss=torch.tensor(0.,device=device)
            ml=torch.zeros(B,T-1,device=device)
            for i,l in enumerate(lengths):
                ml[i,:min(l-1,T-1)]=1.
            xp=model.nh(H[:,:-1,:]); xt=Xf[:,1:,:]
            loss+=(mse(xp,xt).mean(-1)*ml).sum()/(
                ml.sum()+1e-6)
            mf=(torch.rand(B,T,nf,device=device)<mask_rate)
            Xmc=Xp.clone(); Xmc[:,:,:nf][mf]=0.
            Hm=model(Xmc,lengths); xr=model.mh(Hm)
            if mf.sum()>0:
                loss+=mse(xr[:,:,:nf][mf],Xf[mf]).mean()
            ol=[]
            for i in range(B):
                li=lengths[i].item()
                if li<4: continue
                w=max(2,li//3)
                ta=np.random.randint(0,li-w)
                tb=min(np.random.randint(ta+w,li),li-1)
                ha,hb=H[i,ta,:],H[i,tb,:]
                if np.random.random()>0.5:
                    lg=model.oh(
                        torch.cat([ha,hb]).unsqueeze(0))
                    lb=torch.ones(1,device=device)
                else:
                    lg=model.oh(
                        torch.cat([hb,ha]).unsqueeze(0))
                    lb=torch.zeros(1,device=device)
                ol.append(bce(lg.squeeze(),lb.squeeze()))
            if ol: loss+=0.5*torch.stack(ol).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(
                model.parameters(),1.)
            opt.step()
    return model.enc


def mt_pretrain_fold(train_seqs_ext, enc_class, nf,
                      enc_kwargs=None,
                      l1=0.5, l2=1.0, l3=1.0,
                      epochs=30, bs=8,
                      lr=1e-3, wd=1e-4):
    """
    Multi-task pre-training on training fold only.
    enc_class: GRUEncoder | GRUDEncoder
    """
    if enc_kwargs is None: enc_kwargs={}
    enc=enc_class(nf, **enc_kwargs)
    model=MTWrapper(enc, nf, 64).to(device)
    opt=torch.optim.Adam(model.parameters(),
                          lr=lr,weight_decay=wd)
    bce_=nn.BCEWithLogitsLoss(reduction='none')
    ce_ =nn.CrossEntropyLoss(reduction='none')
    mse_=nn.MSELoss(reduction='none')
    dl=DataLoader(PatDSExt(train_seqs_ext),
                  batch_size=bs,shuffle=True,
                  collate_fn=collate_ext)
    for ep in range(epochs):
        model.train()
        for Xp,Yl,Yp,Yn,Ye,lengths in dl:
            Xp=Xp.to(device); B,T,_=Xp.shape
            Yp=Yp.to(device); Yn=Yn.to(device)
            Ye=Ye.to(device)
            opt.zero_grad(); H=model(Xp,lengths)
            loss=torch.tensor(0.,device=device)
            ml=torch.zeros(B,T,device=device)
            for i,l in enumerate(lengths): ml[i,:l]=1.
            if l1>0:
                ev=model.eh(H).squeeze(-1)
                msk=(~torch.isnan(Ye))&(ml.bool())
                if msk.sum()>0:
                    loss+=l1*bce_(ev[msk],Ye[msk]).mean()
            if l2>0:
                ph=model.ph(H.view(B*T,-1))
                pt=Yp.view(-1)
                mf=ml.view(-1).bool()&(pt>=0)
                if mf.sum()>0:
                    loss+=l2*ce_(ph[mf],pt[mf]).mean()
            if l3>0 and T>1:
                cr=model.ch(H[:,:-1,:]).squeeze(-1)
                ct=Yn[:,1:]
                mc=ml[:,:-1].bool()&(~torch.isnan(ct))
                if mc.sum()>0:
                    loss+=l3*mse_(cr[mc],ct[mc]).mean()
            if loss.item()>0:
                loss.backward()
                nn.utils.clip_grad_norm_(
                    model.parameters(),1.)
                opt.step()
    return model.enc

print("✅ Cell 5 done — pre-training functions defined")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 6 — POOLED CV ENGINE with NESTED HYPERPARAMETER TUNING
# Outer loop: participant-level 10-fold CV (unchanged folds, KFold seed 42).
# Inner loop: within each outer training fold, INNER_VAL_FRAC of the training
#   participants are held out as a validation set. Each hyperparameter setting
#   is trained on the remaining training participants and scored by AUC on the
#   validation participants. The best setting is then refitted on the full
#   outer training fold and evaluated once on the outer test fold.
#   Test-fold participants are never used for tuning.
# Deep models: early stopping on the inner validation AUC selects the number
#   of epochs; the refit on the full training fold uses that number.
# SSL models: the pretrained encoder stays frozen; the head is tuned.
# ════════════════════════════════════════════════════════

# ── hyperparameter grids (first entry = previous fixed setting) ──
ML_GRIDS = {
    'rf':  [{'min_samples_leaf': l, 'max_features': f}
            for l in (5, 1, 20) for f in ('sqrt', 0.3)],
    'xgb': [{'max_depth': d, 'learning_rate': r}
            for d in (6, 3) for r in (0.1, 0.05)],
    'lr':  [{'C': c} for c in (1.0, 0.01, 0.1, 10.0)],
    'svm': [{'C': c} for c in (1.0, 0.1, 10.0)],
}
DL_GRID = [{'lr': l, 'drop': d} for l in (1e-3, 3e-4) for d in (0.3, 0.1)]
DL_DEFAULT = DL_GRID[0]


def save_preds(name, task, fold_idx, pids, y_true, y_prob):
    """One CSV per config / task / fold: participant_id, y_true, y_prob."""
    out = pd.DataFrame({'participant_id': np.asarray(pids, dtype=object),
                        'y_true': np.asarray(y_true, dtype=float).astype(int),
                        'y_prob': np.asarray(y_prob, dtype=float)})
    out.to_csv(os.path.join(SAVE_DIR, 'predictions',
               f"{name}__{task}__fold{fold_idx:02d}.csv"), index=False)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)


def safe_auc(y, p):
    try:
        return float(roc_auc_score(np.asarray(y).astype(int), p))
    except Exception:
        return float('nan')


# ── traditional models ──────────────────────────────────
def flat(seqs, task):
    """Entry-level rows. classify: X_t -> y_t ; next: X_t -> y_{t+1}."""
    X, y, p = [], [], []
    for Xs, ys, pid in seqs:
        if task == 'classify':
            X.append(Xs); y.append(ys); p.append(np.repeat(pid, len(ys)))
        elif len(ys) >= 2:
            X.append(Xs[:-1]); y.append(ys[1:]); p.append(np.repeat(pid, len(ys) - 1))
    return np.vstack(X), np.concatenate(y), np.concatenate(p)


def fit_ml(mtype, hp, X, y):
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X); y = y.astype(int)
    pos = max((y == 1).sum(), 1); neg = (y == 0).sum()
    if mtype == 'rf':
        clf = RandomForestClassifier(
            n_estimators=500, max_depth=None, class_weight='balanced',
            min_samples_leaf=hp.get('min_samples_leaf', 5),
            max_features=hp.get('max_features', 'sqrt'),
            n_jobs=-1, random_state=42)
    elif mtype == 'xgb':
        clf = xgb.XGBClassifier(
            n_estimators=500, scale_pos_weight=neg / pos,
            learning_rate=hp.get('learning_rate', 0.1), max_depth=hp.get('max_depth', 6),
            subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
            n_jobs=-1, random_state=42, verbosity=0)
    elif mtype == 'lr':
        clf = LogisticRegression(C=hp.get('C', 1.0), max_iter=1000,
                                 class_weight='balanced', solver='lbfgs', random_state=42)
    elif mtype == 'svm':
        if len(Xs) > 5000:
            idx_ = np.random.default_rng(42).choice(len(Xs), 5000, replace=False)
            Xs, y = Xs[idx_], y[idx_]
        clf = SVC(kernel='rbf', C=hp.get('C', 1.0), gamma='scale',
                  class_weight='balanced', probability=True, random_state=42)
    clf.fit(Xs, y)
    return lambda Xnew: clf.predict_proba(scaler.transform(Xnew))[:, 1]


# ── deep models ─────────────────────────────────────────
def build_encoder(mtype, inp_nf, nf, drop):
    if mtype == 'gru':         return GRUEncoder(inp_nf, 64, drop=drop)
    if mtype == 'lstm':        return LSTMEncoder(inp_nf, 64, drop=drop)
    if mtype == 'grud':        return GRUDEncoder(nf, 64, drop=drop)
    if mtype == 'transformer': return TransformerEncoder(inp_nf, 64, drop=drop)
    if mtype == 'tcn':         return TCNEncoder(inp_nf, 64, drop=drop)
    if mtype == 'gru_tcn':     return GRUTCNEncoder(inp_nf, 64, drop=drop)
    raise ValueError(mtype)


def predict_dl(enc, head, seqs, task):
    enc.eval(); head.eval()
    probs, labels, pids = [], [], []
    with torch.no_grad():
        for X, y, pid in seqs:
            if task == 'next' and len(y) < 2:
                continue
            H = enc(torch.from_numpy(X).float().unsqueeze(0).to(device))
            if task == 'next':
                pr = torch.sigmoid(head(H[:, :-1, :]).squeeze(0)).cpu().numpy(); gt = y[1:]
            else:
                pr = torch.sigmoid(head(H).squeeze(0)).cpu().numpy(); gt = y
            probs.extend(np.atleast_1d(pr).tolist()); labels.extend(gt.tolist())
            pids.extend([pid] * len(gt))
    return np.array(probs), np.array(labels), pids


def train_dl(enc, head, frozen, train_s, task, epochs, lr, wd, bs,
             val_s=None, patience=None):
    """Trains in place. With val_s: early stopping on validation AUC;
    returns (best_auc, best_epoch). Without: returns (nan, epochs)."""
    y_all = np.concatenate([s[1] for s in train_s])
    pw = torch.tensor([(y_all == 0).sum() / max((y_all == 1).sum(), 1)]
                      ).clamp(max=20).to(device)
    lf = nn.BCEWithLogitsLoss(pos_weight=pw, reduction='none')
    params = list(head.parameters()) + ([] if frozen else list(enc.parameters()))
    opt = torch.optim.Adam(params, lr=lr, weight_decay=wd)
    dl = DataLoader(PatDS([(s[0], s[1]) for s in train_s]),
                    batch_size=bs, shuffle=True, collate_fn=collate_v)
    best_auc, best_ep, since = -1.0, epochs, 0
    for ep in range(epochs):
        enc.train(); head.train()
        if frozen: enc.eval()
        for Xp, yp, lengths in dl:
            Xp, yp = Xp.to(device), yp.to(device)
            if frozen:
                with torch.no_grad(): H = enc(Xp, lengths)
            else:
                H = enc(Xp, lengths)
            pr, tg = ((head(H[:, :-1, :]), yp[:, 1:]) if task == 'next' else (head(H), yp))
            mask = ~torch.isnan(tg)
            if mask.sum() == 0: continue
            loss = lf(pr[mask], tg[mask]).mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(params, 1.)
            opt.step()
        if val_s is not None:
            p, l, _ = predict_dl(enc, head, val_s, task)
            a = safe_auc(l, p)
            if a > best_auc: best_auc, best_ep, since = a, ep + 1, 0
            else:
                since += 1
                if patience and since >= patience: break
    return (best_auc, best_ep) if val_s is not None else (float('nan'), epochs)


# ── main CV loop ───────────────────────────────────────
def run_cv(df, model_cfg, n_folds=10, ft_epochs=30, bs=32,
           ssl_epochs=30, ssl_bs=8, lr=1e-3, wd=1e-4, random_seed=42):
    name, feat_cols, mtype = model_cfg['name'], model_cfg['feat_cols'], model_cfg['model_type']
    save_path = f"{SAVE_DIR}/{name}.json"
    results = {'classify': [], 'next': []}
    hparams = {'classify': [], 'next': []}
    start = 0
    if os.path.exists(save_path):
        d = json.load(open(save_path))
        saved = d.get('fold_aucs', {})
        start = min(len(saved.get('classify', [])), len(saved.get('next', [])))
        results = {t: saved.get(t, [])[:start] for t in results}
        hparams = {t: d.get('hparams', {}).get(t, [None] * start)[:start] for t in hparams}
        if start == n_folds:
            print(f"  ✅ Complete: {name}"); return results
        if start > 0: print(f"  Resuming {name} from fold {start+1}")

    nf = len(feat_cols)
    is_grud = mtype in ('grud', 'ssl_grud')
    is_ml = mtype in ('rf', 'xgb', 'lr', 'svm')
    is_ssl = mtype.startswith('ssl_')
    all_seqs = build_sequences_grud(df, feat_cols) if is_grud else build_sequences(df, feat_cols)
    inp_nf = nf * 3 if is_grud else nf

    pid_arr = np.array([s[2] for s in all_seqs])
    splits = list(KFold(n_splits=n_folds, shuffle=True, random_state=random_seed).split(pid_arr))
    fa_path = os.path.join(SAVE_DIR, 'fold_assignment.json')
    if not os.path.exists(fa_path):
        json.dump({str(k): sorted(map(str, pid_arr[te])) for k, (_, te) in enumerate(splits)},
                  open(fa_path, 'w'), indent=1)

    tune = (TUNE == 'all') or (TUNE == 'traditional' and is_ml)
    t_cv = time.time()
    print(f"\n  {name} | {mtype} | {nf} feats | tuning={'on' if tune else 'off'}")

    for fold_idx, (tr_idx, te_idx) in enumerate(splits):
        if fold_idx < start: continue
        set_seed(SEED + fold_idx)
        train_raw = [all_seqs[i] for i in tr_idx]
        test_raw = [all_seqs[i] for i in te_idx]
        train_s, test_s, _, _ = standardize_fold(train_raw, test_raw, nf)

        # inner split of the TRAINING participants (never the test fold)
        perm = np.random.default_rng(SEED + fold_idx).permutation(len(train_raw))
        n_val = max(1, int(round(INNER_VAL_FRAC * len(train_raw))))
        itr_s, ival_s, _, _ = standardize_fold([train_raw[i] for i in perm[n_val:]],
                                               [train_raw[i] for i in perm[:n_val]], nf)

        ssl_enc = None
        if is_ssl:
            enc_cls = {'ssl_gru': GRUEncoder, 'ssl_lstm': LSTMEncoder,
                       'ssl_grud': GRUDEncoder}[mtype]
            ssl_enc = ssl_pretrain_fold(train_s, enc_cls, nf,
                                        enc_kwargs={'hid': 64, 'drop': 0.3, 'n_layers': 2},
                                        epochs=ssl_epochs, bs=ssl_bs, lr=lr, wd=wd)

        for task in ['classify', 'next']:
            if is_ml:
                grid = ML_GRIDS[mtype] if tune else [ML_GRIDS[mtype][0]]
                best = grid[0]
                if len(grid) > 1:
                    Xi, yi, _ = flat(itr_s, task); Xv, yv, _ = flat(ival_s, task)
                    scores = [safe_auc(yv, fit_ml(mtype, hp, Xi, yi)(Xv)) for hp in grid]
                    if not np.all(np.isnan(scores)):   # no lapses in validation -> keep default
                        best = grid[int(np.nanargmax(scores))]
                Xtr, ytr, _ = flat(train_s, task); Xte, yte, pte = flat(test_s, task)
                probs = fit_ml(mtype, best, Xtr, ytr)(Xte)
                save_preds(name, task, fold_idx, pte, yte, probs)
                results[task].append(safe_auc(yte, probs)); hparams[task].append(best)
                continue

            def fresh(hp):
                set_seed(SEED + fold_idx)
                enc = (copy.deepcopy(ssl_enc) if is_ssl
                       else build_encoder(mtype, inp_nf, nf, hp['drop'])).to(device)
                if is_ssl:
                    for p_ in enc.parameters(): p_.requires_grad = False
                return enc, PredHead(64, drop=hp['drop']).to(device)

            grid = DL_GRID if tune else [DL_DEFAULT]
            if tune:
                trials = []
                for hp in grid:
                    enc, head = fresh(hp)
                    a, ep = train_dl(enc, head, is_ssl, itr_s, task, ft_epochs, hp['lr'], wd, bs,
                                     val_s=ival_s, patience=PATIENCE)
                    trials.append((a, ep, hp))
                if all(np.isnan(t[0]) or t[0] < 0 for t in trials):   # no lapses in validation
                    best = dict(DL_DEFAULT, epochs=ft_epochs)
                else:
                    a_best, ep_best, best = max(trials, key=lambda t: np.nan_to_num(t[0], nan=-1))
                    best = dict(best, epochs=int(ep_best))
            else:
                best = dict(DL_DEFAULT, epochs=ft_epochs)

            enc, head = fresh(best)
            train_dl(enc, head, is_ssl, train_s, task, best['epochs'], best['lr'], wd, bs)
            probs, labels, pids_ = predict_dl(enc, head, test_s, task)
            save_preds(name, task, fold_idx, pids_, labels, probs)
            results[task].append(safe_auc(labels, probs)); hparams[task].append(best)

        elapsed = (time.time() - t_cv) / 60
        eta = elapsed / (fold_idx - start + 1) * (n_folds - fold_idx - 1)
        print(f"  Fold {fold_idx+1:2d}/{n_folds}  cls={results['classify'][-1]:.4f}  "
              f"next={results['next'][-1]:.4f}  ETA:{eta:.0f}min")
        json.dump({'fold_aucs': results, 'hparams': hparams, 'model': name,
                   'inductive_ssl': True, 'grud_impl': GRUD_IMPL, 'seed': SEED,
                   'tuning': TUNE if tune else 'off', 'fold_normalisation': True},
                  open(save_path, 'w'), default=str)

    print(f"\n  {name} RESULTS:")
    for task in ['classify', 'next']:
        m, lo, hi = ci95(results[task])
        if m: print(f"  {task:<12} {m:.4f} [{lo:.4f}–{hi:.4f}]")
    return results

print("✅ Cell 6 done — CV engine with nested tuning ready")


## Part B — leakage check

In [ ]:
# ════════════════════════════════════════════════════════
# CELL L1 — LEAKAGE CHECK: entry-level vs participant-level splits
# Run after Part A (Cells 1–6). Same features, same model settings
# (default grid point, no tuning, so the split is the only difference).
#   participant: 10 folds of PARTICIPANTS (the paper's protocol)
#   entry:       10 folds of ENTRIES, shuffled (entries of one person sit in train and test)
# Scaler fitted on the training rows only in both arms (fit_ml does this).
# ════════════════════════════════════════════════════════
LEAK_MODELS = [('RF_exp', 'rf', EXP_FEATS), ('XGB_exp', 'xgb', EXP_FEATS),
               ('RF_orig', 'rf', ORIG_FEATS), ('XGB_orig', 'xgb', ORIG_FEATS)]
LEAK_DIR = os.path.join(SAVE_DIR, 'leakage_check'); os.makedirs(LEAK_DIR, exist_ok=True)

def leak_cv(seqs, mtype, task, split):
    X, y, p = flat(seqs, task)                      # entry rows; next-entry pairs never cross people
    hp = ML_GRIDS[mtype][0]
    if split == 'participant':
        pid_arr = np.array([s[2] for s in seqs])
        folds = [(np.isin(p, pid_arr[tr]), np.isin(p, pid_arr[te]))
                 for tr, te in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(pid_arr)]
    else:
        idx = np.arange(len(y))
        folds = [(np.isin(idx, tr), np.isin(idx, te))
                 for tr, te in KFold(N_FOLDS, shuffle=True, random_state=SEED).split(idx)]
    return [safe_auc(y[te], fit_ml(mtype, hp, X[tr], y[tr])(X[te])) for tr, te in folds]

leak_rows = []
for name, mtype, cols in LEAK_MODELS:
    path = os.path.join(LEAK_DIR, f'{name}.json')
    if os.path.exists(path):
        res = json.load(open(path))
    else:
        seqs = build_sequences(df, cols); res = {}
        for task in ['classify', 'next']:
            for split in ['participant', 'entry']:
                t0 = time.time(); res[f'{task}|{split}'] = leak_cv(seqs, mtype, task, split)
                print(f"  {name:<9} {task:<9} {split:<12} AUC={np.nanmean(res[f'{task}|{split}']):.4f}  ({(time.time()-t0)/60:.1f} min)")
        json.dump(res, open(path, 'w'))
    for task in ['classify', 'next']:
        a_p, a_e = np.array(res[f'{task}|participant']), np.array(res[f'{task}|entry'])
        m_p, lo_p, hi_p = ci95(a_p); m_e, lo_e, hi_e = ci95(a_e)
        leak_rows.append({'model': name, 'task': task,
                          'participant-level AUC': f'{m_p:.3f} [{lo_p:.3f}, {hi_p:.3f}]',
                          'entry-level AUC': f'{m_e:.3f} [{lo_e:.3f}, {hi_e:.3f}]',
                          'inflation': round(m_e - m_p, 3)})
leak_tab = pd.DataFrame(leak_rows)
leak_tab.to_csv(os.path.join(LEAK_DIR, 'leakage_check.csv'), index=False)
print(leak_tab.to_string(index=False))
# Folds are of different units, so compare the means and CIs; a paired test across folds is not valid here.


## Part C — survival rerun (GRU, expanded features)

In [ ]:
# ════════════════════════════════════════════════════════
# CELL S1 — SURVIVAL SETUP: settings, interval grid, targets
# Run after Cells 1–6 of Objective1_Rerun_v2 (data, features, encoders, SSL pretraining, set_seed).
# Same participant folds as the benchmark: KFold(N_FOLDS, shuffle=True, random_state=SEED) over participants.
# ════════════════════════════════════════════════════════
try:
    from lifelines.utils import concordance_index
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'lifelines', '--quiet'])
    from lifelines.utils import concordance_index
from sklearn.metrics import brier_score_loss

SURV_ENC    = 'gru'          # 'gru' or 'grud' (the SSL encoders in the benchmark)
SURV_FEATS  = EXP_FEATS      # expanded features (46)
HORIZONS    = [1, 6, 24, 72] # hours
EDGES       = np.array([0, 1, 3, 6, 12, 24, 48, 72, 168, np.inf])  # interval boundaries (h); every horizon is a boundary
K           = len(EDGES) - 1
H_IDX       = {h: int(np.where(EDGES == h)[0][0]) for h in HORIZONS}  # F(h) = 1 - prod of (1 - hazard) over the first H_IDX[h] intervals
CINDEX_H    = 72             # risk score for the C-index: predicted probability of a lapse within 72 h
MAX_EPOCHS  = 2 if QUICK_TEST else 100
SURV_PATIENCE = 10
SURV_BS, SURV_LR, SURV_WD = 16, 1e-3, 1e-4
REGIMES     = ['scratch', 'ssl_finetuned', 'ssl_frozen']
SURV_DIR    = os.path.join(SAVE_DIR, f'survival_{SURV_ENC}'); os.makedirs(SURV_DIR, exist_ok=True)


def surv_targets(times_h, labels):
    """Per entry t: hours to the next reported lapse at a LATER entry (event=1), otherwise censored at the
    person's last entry (event=0). The last entry has no follow-up and is excluded (keep=False)."""
    n = len(times_h); dur = np.zeros(n, np.float32); ev = np.zeros(n, np.float32); keep = np.ones(n, bool)
    nxt = np.full(n, -1)
    j = -1
    for t in range(n - 1, -1, -1):          # index of the first lapse strictly after t
        nxt[t] = j
        if labels[t] == 1: j = t
    for t in range(n):
        if t == n - 1: keep[t] = False; continue
        if nxt[t] >= 0: dur[t], ev[t] = times_h[nxt[t]] - times_h[t], 1.
        else:           dur[t], ev[t] = times_h[-1] - times_h[t], 0.
    return dur, ev, keep


def interval_targets(dur, ev, keep):
    """Masked targets over the K intervals. Event in interval k*: 0 before, 1 at k*, ignored after.
    Censored at d: 0 for intervals completed before d, ignored from the interval containing d onward."""
    T = len(dur); Y = np.zeros((T, K), np.float32); M = np.zeros((T, K), np.float32)
    lo, hi = EDGES[:-1], EDGES[1:]
    for t in range(T):
        if not keep[t]: continue
        d = dur[t]
        if ev[t] == 1:
            ks = int(np.searchsorted(hi, d, side='right'))   # interval [lo, hi) holding d
            ks = min(ks, K - 1)
            M[t, :ks + 1] = 1.; Y[t, ks] = 1.
        else:
            M[t, hi <= d] = 1.
    return Y, M


def build_surv_data(df, feat_cols, enc):
    """Sequences in the same order as the benchmark builders, plus survival targets and running lapse rate."""
    base = build_sequences_grud(df, feat_cols) if enc == 'grud' else build_sequences(df, feat_cols)
    out = []
    for (X, y, pid), (pid2, g) in zip(base, df.groupby('account_id')):
        assert pid == pid2
        g = g.sort_values('adjusted_craving_record_created').reset_index(drop=True)
        t_h = ((pd.to_datetime(g['adjusted_craving_record_created']) - pd.to_datetime(g['adjusted_craving_record_created']).iloc[0])
               .dt.total_seconds() / 3600).to_numpy(np.float64)
        L = y.astype(int)
        dur, ev, keep = surv_targets(t_h, L)
        Y, M = interval_targets(dur, ev, keep)
        rate = np.cumsum(L) / np.arange(1, len(L) + 1)          # lapse rate up to and including entry t
        out.append(dict(X=X, Y=Y, M=M, dur=dur, ev=ev, keep=keep, rate=rate, pid=pid))
    return out


SDATA = build_surv_data(df, SURV_FEATS, SURV_ENC)
NF = len(SURV_FEATS); INP_NF = NF * 3 if SURV_ENC == 'grud' else NF
_k = np.concatenate([d['keep'] for d in SDATA]); _e = np.concatenate([d['ev'] for d in SDATA])[_k]
_d = np.concatenate([d['dur'] for d in SDATA])[_k]
print(f"{len(SDATA)} participants, {_k.sum():,} entries with follow-up; next lapse observed for {100*_e.mean():.1f}%")
print(f"time to next lapse (events), median {np.median(_d[_e==1]):.1f} h; intervals (h): {EDGES.tolist()}")
for h in HORIZONS:
    obs = ((_e == 1) & (_d < h)) | (_d >= h)
    print(f"  {h:>2} h: {obs.sum():,} entries with observed outcome, {100*((_e==1)&(_d<h))[obs].mean():.1f}% lapse within {h} h")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL S2 — SURVIVAL ENGINE: hazard model, training with early stopping, metrics
# ════════════════════════════════════════════════════════
class HazardModel(nn.Module):
    def __init__(self, enc, hid=64):
        super().__init__(); self.enc = enc; self.head = nn.Linear(hid, K)   # linear hazard head
    def forward(self, x, lengths): return self.head(self.enc(x, lengths))   # logits (B, T, K)


def collate_surv(batch):
    lengths = torch.tensor([len(b['X']) for b in batch]); T = int(lengths.max()); B = len(batch)
    Xp = torch.zeros(B, T, batch[0]['X'].shape[1]); Yp = torch.zeros(B, T, K); Mp = torch.zeros(B, T, K)
    for i, b in enumerate(batch):
        n = len(b['X'])
        Xp[i, :n] = torch.from_numpy(b['X']).float()
        Yp[i, :n] = torch.from_numpy(b['Y']); Mp[i, :n] = torch.from_numpy(b['M'])
    return Xp, Yp, Mp, lengths


def masked_bce(logits, Y, M):
    l = nn.functional.binary_cross_entropy_with_logits(logits, Y, reduction='none')
    return (l * M).sum(), M.sum()


def standardise(train, other_lists):
    """Means and SDs from TRAINING participants only (first NF columns; GRU-D mask/delta columns untouched)."""
    A = np.vstack([d['X'][:, :NF] for d in train]); m, s = A.mean(0), A.std(0) + 1e-6
    def norm(d):
        X = d['X'].copy(); X[:, :NF] = (X[:, :NF] - m) / s; return dict(d, X=X)
    return [norm(d) for d in train], [[norm(d) for d in L] for L in other_lists]


def eval_loss(model, data):
    model.eval(); tot, cnt = 0., 0.
    with torch.no_grad():
        for Xp, Yp, Mp, lengths in DataLoader(data, batch_size=SURV_BS, collate_fn=collate_surv):
            s, c = masked_bce(model(Xp.to(device), lengths), Yp.to(device), Mp.to(device)); tot += s.item(); cnt += c.item()
    return tot / max(cnt, 1)


def train_surv(model, tr, va, frozen=False):
    """Train on tr; keep the checkpoint with the lowest validation loss on va (validation participants only)."""
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=SURV_LR, weight_decay=SURV_WD)
    best, best_ep, best_state, wait, hist = np.inf, 0, None, 0, []
    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        if frozen: model.enc.eval()                               # frozen encoder: no dropout, no updates
        for Xp, Yp, Mp, lengths in DataLoader(tr, batch_size=SURV_BS, shuffle=True, collate_fn=collate_surv):
            opt.zero_grad(); s, c = masked_bce(model(Xp.to(device), lengths), Yp.to(device), Mp.to(device))
            if c == 0: continue
            (s / c).backward(); nn.utils.clip_grad_norm_(params, 1.0); opt.step()
        v = eval_loss(model, va); hist.append(v)
        if v < best - 1e-5: best, best_ep, best_state, wait = v, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= SURV_PATIENCE: break
    model.load_state_dict(best_state)
    return best_ep, hist


def predict_surv(model, data):
    """F(h) = P(lapse within h hours) for every entry with follow-up."""
    model.eval(); rows = []
    with torch.no_grad():
        for d in data:
            haz = torch.sigmoid(model(torch.from_numpy(d['X']).float().unsqueeze(0).to(device),
                                      torch.tensor([len(d['X'])]))).squeeze(0).cpu().numpy()   # (T, K)
            S = np.cumprod(1 - haz, axis=1)
            k = d['keep']
            r = pd.DataFrame({'pid': d['pid'], 'entry': np.arange(len(k))[k], 'dur': d['dur'][k], 'event': d['ev'][k],
                              'rate': d['rate'][k]})
            for h in HORIZONS: r[f'F{h}'] = 1 - S[k, H_IDX[h] - 1]
            rows.append(r)
    return pd.concat(rows, ignore_index=True)


def surv_metrics(P, score_col=None):
    """C-index (Harrell) and, per horizon, AUC and Brier on entries whose outcome within h is observed."""
    sc = P[score_col] if score_col else P[f'F{CINDEX_H}']
    out = {'cindex': concordance_index(P.dur, -sc, P.event)}     # higher risk -> shorter time
    for h in HORIZONS:
        obs = ((P.event == 1) & (P.dur < h)) | (P.dur >= h)
        y = (((P.event == 1) & (P.dur < h))[obs]).astype(int); p = (P[score_col] if score_col else P[f'F{h}'])[obs]
        out[f'auc{h}'] = roc_auc_score(y, p) if y.nunique() == 2 else np.nan
        out[f'brier{h}'] = brier_score_loss(y, p) if not score_col else np.nan   # running rate is not a probability of lapse within h
        out[f'n{h}'], out[f'prev{h}'] = int(obs.sum()), float(y.mean())
    return out
print("✅ Cell S2 done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL S3 — RUN: 10 participant folds × 3 encoder regimes (+ running-lapse-rate baseline)
# Per fold: standardise on training participants → SSL pretraining on training participants (shared by the two SSL
# regimes) → inner split of training participants (80% fit / 20% early stopping, same rule as the benchmark) →
# train each regime, keep its best-validation checkpoint → predict the test participants once.
# Resumable: each fold's results are saved; rerunning skips finished folds.
# ════════════════════════════════════════════════════════
ENC_CLS = GRUDEncoder if SURV_ENC == 'grud' else GRUEncoder
pid_arr = np.array([d['pid'] for d in SDATA])
splits = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(pid_arr))
res_path = os.path.join(SURV_DIR, 'fold_metrics.csv')
done = pd.read_csv(res_path) if os.path.exists(res_path) else pd.DataFrame()
t0 = time.time()
for k, (tr_idx, te_idx) in enumerate(splits):
    if len(done) and (done.fold == k).sum() >= len(REGIMES) + 1: continue
    set_seed(SEED + k)
    train_raw = [SDATA[i] for i in tr_idx]; test_raw = [SDATA[i] for i in te_idx]
    train_s, (test_s,) = standardise(train_raw, [test_raw])
    perm = np.random.default_rng(SEED + k).permutation(len(train_s))
    n_val = max(1, int(round(INNER_VAL_FRAC * len(train_s))))
    itr, ival = [train_s[i] for i in perm[n_val:]], [train_s[i] for i in perm[:n_val]]

    enc_path = os.path.join(SURV_DIR, f'ssl_encoder_fold{k:02d}.pt')
    ssl_enc = ENC_CLS(NF, hid=64, drop=0.3, n_layers=2).to(device)
    if os.path.exists(enc_path):
        ssl_enc.load_state_dict(torch.load(enc_path, map_location=device))
    else:
        set_seed(SEED + k)
        ssl_enc = ssl_pretrain_fold([(d['X'], np.zeros(len(d['X']), np.float32), d['pid']) for d in train_s], ENC_CLS, NF,
                                    enc_kwargs={'hid': 64, 'drop': 0.3, 'n_layers': 2}, epochs=SSL_EPOCHS, bs=8, lr=1e-3, wd=1e-4)
        torch.save(ssl_enc.state_dict(), enc_path)

    rows = []
    for regime in REGIMES:
        set_seed(SEED + k)
        if regime == 'scratch': enc = ENC_CLS(NF, hid=64, drop=0.3, n_layers=2)
        else:                   enc = copy.deepcopy(ssl_enc)
        model = HazardModel(enc.to(device)).to(device)
        if regime == 'ssl_frozen':
            for p in model.enc.parameters(): p.requires_grad = False
        best_ep, hist = train_surv(model, itr, ival, frozen=(regime == 'ssl_frozen'))
        P = predict_surv(model, test_s); P.to_csv(os.path.join(SURV_DIR, f'pred_{regime}_fold{k:02d}.csv'), index=False)
        m = surv_metrics(P); rows.append(dict(fold=k, regime=regime, best_epoch=best_ep, epochs_run=len(hist), **m))
        json.dump(hist, open(os.path.join(SURV_DIR, f'valhist_{regime}_fold{k:02d}.json'), 'w'))
        print(f"  fold {k+1:2d} {regime:<14} C={m['cindex']:.3f}  AUC1/6/24/72="
              f"{m['auc1']:.3f}/{m['auc6']:.3f}/{m['auc24']:.3f}/{m['auc72']:.3f}  best epoch {best_ep}/{len(hist)}")
    m = surv_metrics(P, score_col='rate'); rows.append(dict(fold=k, regime='running_rate', best_epoch=np.nan, epochs_run=np.nan, **m))
    print(f"  fold {k+1:2d} {'running_rate':<14} C={m['cindex']:.3f}  ({(time.time()-t0)/60:.0f} min so far)")
    done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_csv(res_path, index=False)
print("✅ Cell S3 done")


In [ ]:
# ════════════════════════════════════════════════════════
# CELL S3b — FROZEN REGIME AGAIN, WITHOUT THE 100-EPOCH LIMIT
# In S3 the frozen regime's best validation loss fell at the 100-epoch limit in nearly every fold, so it was still
# improving. With the encoder frozen (and in eval mode), its output for each entry never changes, so it is computed
# ONCE and the linear hazard head is trained on it — identical to S3's frozen training, only much faster.
# Same folds, same saved SSL encoders, same inner validation participants; only the epoch limit is raised.
# Writes fold_metrics.csv with the frozen rows replaced (the old file is kept as fold_metrics_frozen100.csv).
# Run after S1 and S2 (and after S3, which saved the encoders). Then rerun S4.
# ════════════════════════════════════════════════════════
FROZEN_MAX_EPOCHS = 2000
FROZEN_PATIENCE   = 30

class _Cached(nn.Module):                       # passes the cached encoder output straight to the hazard head
    def forward(self, x, lengths=None): return x

def _embed(enc, data):
    enc.eval(); out = []
    with torch.no_grad():
        for d in data:
            H = enc(torch.from_numpy(d['X']).float().unsqueeze(0).to(device), torch.tensor([len(d['X'])]))[0].cpu().numpy()
            out.append(dict(d, X=H.astype(np.float32)))
    return out

ENC_CLS = GRUDEncoder if SURV_ENC == 'grud' else GRUEncoder
pid_arr = np.array([d['pid'] for d in SDATA])
splits = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(pid_arr))
res_path = os.path.join(SURV_DIR, 'fold_metrics.csv')
backup = os.path.join(SURV_DIR, 'fold_metrics_frozen100.csv')
R_old = pd.read_csv(backup if os.path.exists(backup) else res_path)
if not os.path.exists(backup): R_old.to_csv(backup, index=False)

_saved_max = MAX_EPOCHS; MAX_EPOCHS = FROZEN_MAX_EPOCHS
_saved_pat = SURV_PATIENCE; SURV_PATIENCE = FROZEN_PATIENCE
new_rows = []; t0 = time.time()
for k, (tr_idx, te_idx) in enumerate(splits):
    set_seed(SEED + k)
    train_s, (test_s,) = standardise([SDATA[i] for i in tr_idx], [[SDATA[i] for i in te_idx]])
    perm = np.random.default_rng(SEED + k).permutation(len(train_s))
    n_val = max(1, int(round(INNER_VAL_FRAC * len(train_s))))
    itr, ival = [train_s[i] for i in perm[n_val:]], [train_s[i] for i in perm[:n_val]]
    enc = ENC_CLS(NF, hid=64, drop=0.3, n_layers=2).to(device)
    enc.load_state_dict(torch.load(os.path.join(SURV_DIR, f'ssl_encoder_fold{k:02d}.pt'), map_location=device))
    E_itr, E_val, E_te = _embed(enc, itr), _embed(enc, ival), _embed(enc, test_s)
    set_seed(SEED + k)
    model = HazardModel(_Cached()).to(device)
    best_ep, hist = train_surv(model, E_itr, E_val)
    P = predict_surv(model, E_te); P.to_csv(os.path.join(SURV_DIR, f'pred_ssl_frozen_long_fold{k:02d}.csv'), index=False)
    json.dump(hist, open(os.path.join(SURV_DIR, f'valhist_ssl_frozen_long_fold{k:02d}.json'), 'w'))
    m = surv_metrics(P); new_rows.append(dict(fold=k, regime='ssl_frozen', best_epoch=best_ep, epochs_run=len(hist), **m))
    old = R_old[(R_old.fold == k) & (R_old.regime == 'ssl_frozen')].iloc[0]
    print(f"  fold {k+1:2d} frozen  C={m['cindex']:.3f} (was {old.cindex:.3f})  AUC72={m['auc72']:.3f} (was {old.auc72:.3f})"
          f"  best epoch {best_ep}/{len(hist)}  ({(time.time()-t0)/60:.1f} min)")
MAX_EPOCHS, SURV_PATIENCE = _saved_max, _saved_pat

R_new = pd.concat([R_old[R_old.regime != 'ssl_frozen'], pd.DataFrame(new_rows)], ignore_index=True).sort_values(['fold', 'regime'])
R_new.to_csv(res_path, index=False)
print(f"\nFrozen regime, mean over folds: C-index {R_old[R_old.regime=='ssl_frozen'].cindex.mean():.3f} (100-epoch limit) → "
      f"{pd.DataFrame(new_rows).cindex.mean():.3f}; best epoch median {pd.DataFrame(new_rows).best_epoch.median():.0f}")
print("✅ fold_metrics.csv updated — now rerun S4")

In [ ]:
# ════════════════════════════════════════════════════════
# CELL S4 — RESULTS: Table (mean (SD) over folds), paired comparisons with Holm correction, early-stopping check
# ════════════════════════════════════════════════════════
R = pd.read_csv(os.path.join(SURV_DIR, 'fold_metrics.csv'))
NAMES = {'scratch': 'From scratch', 'ssl_finetuned': 'SSL fine-tuned', 'ssl_frozen': 'SSL frozen', 'running_rate': 'Running lapse rate (baseline)'}
METRICS = ['cindex'] + [f'auc{h}' for h in HORIZONS] + [f'brier{h}' for h in HORIZONS]
tab = []
for reg in ['scratch', 'ssl_finetuned', 'ssl_frozen', 'running_rate']:
    d = R[R.regime == reg]; row = {'Regime': NAMES[reg], 'folds': len(d)}
    for c in METRICS:
        row[c] = '–' if d[c].isna().all() else f"{d[c].mean():.3f} ({d[c].std(ddof=1):.3f})"
    tab.append(row)
tab = pd.DataFrame(tab); tab.to_csv(os.path.join(SURV_DIR, 'table_survival.csv'), index=False)
print("Survival performance, mean (SD) over folds"); print(tab.to_string(index=False))

# paired fold-level comparisons against training from scratch
cmp = []
for reg in ['ssl_finetuned', 'ssl_frozen']:
    for c in METRICS:
        a = R[R.regime == reg].set_index('fold')[c]; b = R[R.regime == 'scratch'].set_index('fold')[c]
        dlt = (a - b).dropna(); m, lo, hi = ci95(dlt.values)
        p = stats.wilcoxon(dlt).pvalue if (dlt != 0).sum() >= 1 else 1.0
        cmp.append(dict(comparison=f'{NAMES[reg]} − From scratch', metric=c, mean_diff=m, ci_lo=lo, ci_hi=hi, p_wilcoxon=p))
cmp = pd.DataFrame(cmp)
# Holm across all 18 comparisons (2 regimes × [C-index + 4 AUC + 4 Brier])
order = np.argsort(cmp.p_wilcoxon.values); m_ = len(cmp); adj = np.empty(m_); run = 0
for r, i in enumerate(order):
    run = max(run, min(1, (m_ - r) * cmp.p_wilcoxon.values[i])); adj[i] = run
cmp['p_holm'] = adj
cmp.to_csv(os.path.join(SURV_DIR, 'paired_comparisons.csv'), index=False)
pd.set_option('display.float_format', '{:.4f}'.format)
print("\nPaired differences vs training from scratch (fold level; 95% CI from t; Wilcoxon signed-rank; Holm over 18 tests)")
print(cmp.to_string(index=False))
print("\nNote: with 10 folds the smallest possible two-sided Wilcoxon p is 0.002; Holm over 18 tests needs p < 0.0028 for the top test.")

# early stopping check: the epoch kept and whether training stopped before the cap
es = R[R.regime != 'running_rate'].groupby('regime')[['best_epoch', 'epochs_run']].agg(['median', 'min', 'max'])
print(f"\nEpoch kept (best validation loss) and epochs run, cap = {MAX_EPOCHS}:"); print(es)
print("\nEvaluable entries and lapse prevalence per horizon (fold mean):")
print(R[R.regime == 'scratch'][[f'n{h}' for h in HORIZONS] + [f'prev{h}' for h in HORIZONS]].mean().round(3).to_string())
